In [0]:
import uuid
from pyspark.sql import functions as F
from pyspark.sql.types import *

In [0]:

# database schema
CATALOG_SCHEMA = "dev.electroflow_pipeline."

# bronze tables
bronze_customers_table = CATALOG_SCHEMA + "bronze_customers"
bronze_products_table = CATALOG_SCHEMA + "bronze_products"
bronze_orders_table = CATALOG_SCHEMA + "bronze_orders"
bronze_payments_table = CATALOG_SCHEMA + "bronze_payments"
bronze_coupons_table = CATALOG_SCHEMA + "bronze_coupons"

# silver tables
silver_customers_table = CATALOG_SCHEMA + "silver_customers"
silver_products_table = CATALOG_SCHEMA + "silver_products"
silver_orders_table = CATALOG_SCHEMA + "silver_orders"
silver_order_items_table = CATALOG_SCHEMA + "silver_order_items"
silver_payments_table = CATALOG_SCHEMA + "silver_payments"
silver_coupons_table = CATALOG_SCHEMA + "silver_coupons"

__functions

In [0]:
#customers
def transform_customers():
    print(f"--- cleaning customers...")

    # load data from bronze
    df = spark.table(bronze_customers_table)

    # Clean & Refine
    # - Deduplicate by customer_id
    # - Handle mixed date formats (yyyy-MM-dd and MM/dd/yyyy)
    # - Fill missing phone numbers
    df_clean = df.dropDuplicates(["customer_id"])\
        .withColumn("join_date", F.coalesce(
            F.try_to_date("join_date", "yyyy-MM-dd"), 
            F.try_to_date("join_date", "MM/dd/yyyy")
        )) \
        .withColumn("phone_number", F.coalesce(F.col("phone_number"), F.lit("unknown")))\
        .withColumn("gender_int", 
            F.when(F.col("gender") == "male", 1)
             .when(F.col("gender") == "female", 2)
             .otherwise(0))
    print("cleaning done")
    
    #write to silver column
    df_clean.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(silver_customers_table)
    print("writing customers done")
    return df_clean

In [0]:

#orders and order items
def transform_orders():
    print("🛒 Cleaning Orders and Items...")

    #load
    df = spark.table(bronze_orders_table)

    # --- 1. Silver Orders (Headers) ---
    # Drop the items array to keep the table normalized in 3NF
    df_orders = df.dropDuplicates(["order_id"]).drop("items")
    
    df_orders.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(silver_orders_table)
    print("writing orders done")

    # --- 2. Silver Order Items (Line Items) ---
    # Explode the items array into individual rows
    df_items = df.select("order_id", F.explode("items").alias("item"))\
        .select(
            "order_id",
            F.col("item.product_id").alias("product_id"),
            F.col("item.quantity").alias("quantity"),
            F.col("item.unit_price").alias("unit_price").cast("decimal(18,2)"),
            F.col("item.item_total").alias("item_total").cast("decimal(18,2)")
        )
        
    # Generate a unique key for order items if needed, but order_id + product_id should suffice
    df_items = df_items.dropDuplicates(["order_id", "product_id"])
    
    df_items.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(silver_order_items_table)
    print("writing order items done")
    
    return df_orders, df_items

In [0]:

#products
def transform_products():
    print("📦 Cleaning Products...")
    
    # Load
    df = spark.table(bronze_products_table)
    
    # Clean & Refine
    # - Deduplicate by product_id
    # - Ensure price is a decimal for calculation accuracy
    df_clean = df.dropDuplicates(["product_id"]) \
        .withColumn("price", F.col("base_price").cast("decimal(18,2)"))
    print("cleaning products is done done")

    # Write
    df_clean.write.format("delta").mode("overwrite").saveAsTable(silver_products_table)
    print("writing products done")
    return df_clean

In [0]:

#payments
def transform_payments():
    print("💳 Cleaning Payments...")
    
    # Load
    df = spark.table(bronze_payments_table)
    
    # Clean & Refine
    df_clean = df.dropDuplicates(["payment_id"]) \
        .withColumn("payment_value", F.col("payment_value").cast("decimal(18,2)"))
    
    # Write
    df_clean.write.format("delta").mode("overwrite").saveAsTable(silver_payments_table)
    print("writing payments done")
    return df_clean

In [0]:

#coupons
def transform_coupons():
    print("🏷️ Cleaning Coupons...")
    
    # Load
    df = spark.table(bronze_coupons_table)
    
    # Clean & Refine
    df_clean = df.dropDuplicates(["coupon_code"])
    
    # Write
    df_clean.write.format("delta").mode("overwrite").saveAsTable(silver_coupons_table)
    print("writing coupons done")
    return df_clean

In [0]:

#---execute the functions
silver_customers = transform_customers()
silver_orders, silver_order_items = transform_orders()
silver_products = transform_products()
silver_payments = transform_payments()
silver_coupons = transform_coupons()
print("✅ All transformations complete.")